<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Finance-Decision-Copilot1.1/blob/main/Enterprise_Finance_Decision_Copilot1_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 1: INSTALL PACKAGES
# ============================================================

!pip -q install pandas numpy faker scikit-learn sentence-transformers pyarrow openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.0 MB/s eta 0:00:00


In [2]:
# ============================================================
# STEP 2: IMPORTS AND PROJECT FOLDERS
# ============================================================

import os
import json
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

PROJECT_NAME = "enterprise-finance-decision-copilot"
PROJECT_DIR = Path(PROJECT_NAME)

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, OUTPUT_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project path:", PROJECT_DIR.resolve())

Project folders created successfully.
Project path: /content/enterprise-finance-decision-copilot


In [3]:
# ============================================================
# STEP 3: PROJECT CONFIG
# ============================================================

config = {
    "project_name": "Enterprise Finance Decision Copilot",
    "domain": "SAP FICO style finance intelligence",
    "objective": "Generate enterprise finance data, run DQ checks, build decision logic, create semantic search and decision copilot output.",
    "version": "1.0.0",
    "author": "Ranjith",
    "created_at": datetime.now().isoformat()
}

with open(CONFIG_DIR / "project_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Config file saved successfully.")
print(json.dumps(config, indent=2))

Config file saved successfully.
{
  "project_name": "Enterprise Finance Decision Copilot",
  "domain": "SAP FICO style finance intelligence",
  "objective": "Generate enterprise finance data, run DQ checks, build decision logic, create semantic search and decision copilot output.",
  "version": "1.0.0",
  "author": "Ranjith",
  "created_at": "2026-04-06T04:44:27.116061"
}


In [4]:
# ============================================================
# STEP 4: CREATE MASTER DATA
# ============================================================

vendor_categories = [
    "Maintenance Services",
    "IT Services",
    "Electrical Components",
    "Field Operations",
    "Logistics",
    "Consulting",
    "Inspection Services"
]

vendor_states = ["NY", "NJ", "PA", "CT", "MA", "MI", "IL", "TX"]

gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["400200", "Revenue - Grid Services", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["500500", "Opex - Compliance", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])

cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Operations"],
    ["CC140", "Finance Operations", "Finance"],
    ["CC150", "Supplier Quality", "Quality"],
    ["CC160", "Asset Reliability", "Reliability"]
], columns=["cost_center_id", "cost_center_name", "department"])

invoice_descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection",
    "compliance audit service",
    "urgent material replacement",
    "vendor maintenance support"
]

policy_topics = [
    "invoice approval threshold",
    "duplicate invoice control",
    "late payment escalation",
    "vendor compliance review",
    "high amount approval workflow",
    "unmatched purchase order handling"
]

print("Master data references ready.")

Master data references ready.


In [5]:
# ============================================================
# STEP 5: GENERATE VENDOR MASTER DATA
# ============================================================

vendor_rows = []

for i in range(1, 51):
    vendor_id = f"V{i:03d}"
    vendor_name = fake.company()
    vendor_category = random.choice(vendor_categories)
    vendor_state = random.choice(vendor_states)
    vendor_risk_score = round(random.uniform(0.05, 0.95), 2)
    preferred_vendor_flag = random.choice(["Y", "N"])

    vendor_rows.append([
        vendor_id,
        vendor_name,
        vendor_category,
        vendor_state,
        vendor_risk_score,
        preferred_vendor_flag
    ])

vendors = pd.DataFrame(vendor_rows, columns=[
    "vendor_id",
    "vendor_name",
    "vendor_category",
    "vendor_state",
    "vendor_risk_score",
    "preferred_vendor_flag"
])

vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)

print("Master data saved.")
display(vendors.head())

Master data saved.


,vendor_id,vendor_name,vendor_category,vendor_state,vendor_risk_score,preferred_vendor_flag
0,V001,"Rodriguez, Figueroa and Sanchez",Consulting,NJ,0.07,N
1,V002,Doyle Ltd,IT Services,CT,0.18,Y
2,V003,"Mcclain, Miller and Henderson",Consulting,NJ,0.58,Y
3,V004,Davis and Sons,Maintenance Services,NJ,0.25,Y
4,V005,"Guzman, Hoffman and Baldwin",Logistics,CT,0.69,N


In [6]:
# ============================================================
# STEP 6: GENERATE TRANSACTION AND POLICY DATA
# ============================================================

start_date = datetime(2024, 1, 1)

invoice_rows = []
payment_rows = []
po_rows = []
policy_rows = []
case_rows = []

approval_statuses = ["Approved", "Pending", "Rejected", "Under Review"]
payment_statuses = ["Paid", "Partially Paid", "Pending"]
invoice_sources = ["SAP", "Ariba", "Coupa", "Manual Upload"]
business_units = ["Utility Finance", "Grid Ops", "Shared Services", "Procurement"]

# Purchase Orders
for i in range(1, 121):
    po_id = f"PO{i:05d}"
    vendor = vendors.sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]
    po_date = start_date + timedelta(days=random.randint(0, 365))
    po_amount = round(random.uniform(5000, 150000), 2)

    po_rows.append([
        po_id,
        vendor["vendor_id"],
        cc["cost_center_id"],
        po_date.date().isoformat(),
        po_amount,
        random.choice(["Open", "Closed", "Released"]),
        random.choice(business_units)
    ])

purchase_orders = pd.DataFrame(po_rows, columns=[
    "po_id", "vendor_id", "cost_center_id", "po_date",
    "po_amount_usd", "po_status", "business_unit"
])

# Invoices and Payments
for i in range(1, 301):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]
    po = purchase_orders.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 15))
    due_date = posting_date + timedelta(days=random.randint(15, 45))
    amount = round(random.uniform(1000, 85000), 2)

    invoice_id = f"INV{i:05d}"
    source_doc_id = f"SRC{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        po["po_id"],
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        due_date.date().isoformat(),
        amount,
        "USD",
        random.choice(invoice_descriptions),
        random.choice(approval_statuses),
        random.choice(invoice_sources),
        random.choice(business_units)
    ])

    payment_status = random.choice(payment_statuses)
    if payment_status == "Paid":
        paid_amount = round(amount * random.uniform(0.97, 1.00), 2)
        payment_date = due_date + timedelta(days=random.randint(-5, 20))
    elif payment_status == "Partially Paid":
        paid_amount = round(amount * random.uniform(0.30, 0.80), 2)
        payment_date = due_date + timedelta(days=random.randint(0, 25))
    else:
        paid_amount = 0.0
        payment_date = due_date + timedelta(days=random.randint(10, 35))

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        paid_amount,
        payment_status
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "po_id", "vendor_id", "gl_account",
    "cost_center_id", "invoice_date", "posting_date", "due_date",
    "amount_usd", "currency", "description", "approval_status",
    "invoice_source", "business_unit"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id",
    "payment_date", "paid_amount_usd", "payment_status"
])

# Policies
for i in range(1, 31):
    topic = random.choice(policy_topics)
    threshold = random.choice([5000, 10000, 25000, 50000])

    policy_rows.append([
        f"POL{i:03d}",
        topic,
        f"{topic.title()} policy requires validation for invoices above {threshold} USD and mandatory review if vendor risk is high or PO match is missing.",
        random.choice(["Finance Controls", "Procurement Governance", "AP Operations"]),
        random.choice(["Low", "Medium", "High"])
    ])

policies = pd.DataFrame(policy_rows, columns=[
    "policy_id", "policy_topic", "policy_text", "policy_owner", "severity"
])

# Investigation Cases
case_reasons = [
    "Duplicate invoice suspicion",
    "High-risk vendor review",
    "Late payment follow-up",
    "Missing PO validation",
    "Amount threshold escalation",
    "Compliance exception check"
]

for i in range(1, 81):
    case_rows.append([
        f"CASE{i:04d}",
        random.choice(case_reasons),
        random.choice(invoices["invoice_id"].tolist()),
        random.choice(["Open", "Closed", "In Progress"]),
        random.choice(["Low", "Medium", "High"]),
        fake.sentence(nb_words=10)
    ])

investigation_cases = pd.DataFrame(case_rows, columns=[
    "case_id", "case_reason", "invoice_id", "case_status", "case_priority", "case_summary"
])

purchase_orders.to_csv(RAW_DIR / "purchase_orders.csv", index=False)
invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)
policies.to_csv(RAW_DIR / "policies.csv", index=False)
investigation_cases.to_csv(RAW_DIR / "investigation_cases.csv", index=False)

print("Transaction and policy datasets saved.")
display(invoices.head())

Transaction and policy datasets saved.


,invoice_id,source_doc_id,po_id,vendor_id,gl_account,cost_center_id,invoice_date,posting_date,due_date,amount_usd,currency,description,approval_status,invoice_source,business_unit
0,INV00001,SRC00001,PO00098,V043,500200,CC140,2024-10-10,2024-10-11,2024-11-11,80637.26,USD,sap fico support work,Pending,Manual Upload,Procurement
1,INV00002,SRC00002,PO00058,V020,400200,CC140,2024-04-21,2024-05-03,2024-06-02,55255.84,USD,compliance audit service,Under Review,SAP,Utility Finance
2,INV00003,SRC00003,PO00021,V019,500100,CC140,2024-04-29,2024-05-04,2024-06-09,49850.67,USD,compliance audit service,Rejected,Manual Upload,Shared Services
3,INV00004,SRC00004,PO00089,V049,400100,CC130,2024-05-31,2024-06-05,2024-06-27,11135.09,USD,substation maintenance work,Rejected,SAP,Grid Ops
4,INV00005,SRC00005,PO00013,V017,500200,CC100,2024-05-09,2024-05-21,2024-06-23,83034.99,USD,urgent material replacement,Rejected,SAP,Grid Ops


In [7]:
# ============================================================
# STEP 7: INJECT REALISTIC DATA ISSUES
# ============================================================

invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500.00
invoices.loc[10, "description"] = None
invoices.loc[15, "amount_usd"] = 175000.00
invoices.loc[15, "approval_status"] = "Pending"
invoices.loc[20, ["vendor_id", "gl_account", "cost_center_id", "amount_usd", "description"]] = \
    invoices.loc[21, ["vendor_id", "gl_account", "cost_center_id", "amount_usd", "description"]].values
invoices.loc[25, "po_id"] = None

payments.loc[30, "payment_status"] = "Pending"
payments.loc[30, "paid_amount_usd"] = 0.0

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Issues injected successfully.")

Issues injected successfully.


In [8]:
# ============================================================
# STEP 8: LOAD RAW FILES AND BUILD INTEGRATED VIEW
# ============================================================

vendors_df = pd.read_csv(RAW_DIR / "vendors.csv")
gl_accounts_df = pd.read_csv(RAW_DIR / "gl_accounts.csv")
cost_centers_df = pd.read_csv(RAW_DIR / "cost_centers.csv")
purchase_orders_df = pd.read_csv(RAW_DIR / "purchase_orders.csv")
invoices_df = pd.read_csv(RAW_DIR / "invoices.csv")
payments_df = pd.read_csv(RAW_DIR / "payments.csv")
policies_df = pd.read_csv(RAW_DIR / "policies.csv")
cases_df = pd.read_csv(RAW_DIR / "investigation_cases.csv")

integrated_df = (
    invoices_df
    .merge(vendors_df, on="vendor_id", how="left")
    .merge(gl_accounts_df, on="gl_account", how="left")
    .merge(cost_centers_df, on="cost_center_id", how="left")
    .merge(purchase_orders_df[["po_id", "po_amount_usd", "po_status"]], on="po_id", how="left")
    .merge(payments_df, on=["invoice_id", "source_doc_id"], how="left")
)

print("Integrated dataframe created.")
print("Shape:", integrated_df.shape)
display(integrated_df.head())

Integrated dataframe created.
Shape: (300, 30)


,invoice_id,source_doc_id,po_id,vendor_id,gl_account,cost_center_id,invoice_date,posting_date,due_date,amount_usd,...,gl_account_name,gl_type,cost_center_name,department,po_amount_usd,po_status,payment_id,payment_date,paid_amount_usd,payment_status
0,INV00001,SRC00001,PO00098,V043,500200,CC140,2024-10-10,2024-10-11,2024-11-11,80637.26,...,Opex - IT Services,Expense,Finance Operations,Finance,73194.49,Released,PAY00001,2024-12-15,0.00,Pending
1,INV00002,SRC00002,PO00058,V020,400200,CC140,2024-04-21,2024-05-03,2024-06-02,55255.84,...,Revenue - Grid Services,Revenue,Finance Operations,Finance,107773.87,Open,PAY00002,2024-06-24,22699.00,Partially Paid
2,INV00003,SRC00003,PO00021,V019,500100,CC140,2024-04-29,2024-05-04,2024-06-09,49850.67,...,Opex - Maintenance,Expense,Finance Operations,Finance,100319.11,Released,PAY00003,2024-06-17,32474.10,Partially Paid
3,INV00004,SRC00004,PO00089,NaN,400100,CC130,2024-05-31,2024-06-05,2024-06-27,11135.09,...,Revenue - Energy Delivery,Revenue,Customer Support,Customer Operations,114253.89,Closed,PAY00004,2024-07-07,10873.32,Paid
4,INV00005,SRC00005,PO00013,V017,500200,CC100,2024-05-09,2024-05-21,2024-06-23,83034.99,...,Opex - IT Services,Expense,Transmission Operations,Operations,106005.77,Released,PAY00005,2024-06-28,34355.03,Partially Paid


In [9]:
# ============================================================
# STEP 9: STANDARDIZATION AND COMMON BUSINESS KEY
# ============================================================

integrated_df["vendor_name_clean"] = integrated_df["vendor_name"].fillna("").str.strip().str.upper()
integrated_df["description_clean"] = integrated_df["description"].fillna("").str.strip().str.upper()
integrated_df["cost_center_name_clean"] = integrated_df["cost_center_name"].fillna("").str.strip().str.upper()
integrated_df["gl_account_name_clean"] = integrated_df["gl_account_name"].fillna("").str.strip().str.upper()

integrated_df["common_business_key"] = (
    integrated_df["vendor_id"].fillna("MISSING_VENDOR").astype(str) + "_" +
    integrated_df["cost_center_id"].fillna("MISSING_CC").astype(str) + "_" +
    integrated_df["gl_account"].fillna("MISSING_GL").astype(str)
)

print("Common business key generated.")
display(integrated_df[["invoice_id", "common_business_key"]].head())

Common business key generated.


,invoice_id,common_business_key
0,INV00001,V043_CC140_500200
1,INV00002,V020_CC140_400200
2,INV00003,V019_CC140_500100
3,INV00004,MISSING_VENDOR_CC130_400100
4,INV00005,V017_CC100_500200


In [10]:
# ============================================================
# STEP 10: DATA QUALITY CHECKS
# ============================================================

dq_records = []

for _, row in integrated_df.iterrows():
    if pd.isna(row["vendor_id"]):
        dq_records.append([row["invoice_id"], "Missing vendor_id"])
    if pd.notna(row["amount_usd"]) and float(row["amount_usd"]) <= 0:
        dq_records.append([row["invoice_id"], "Invalid amount"])
    if pd.isna(row["description"]) or str(row["description"]).strip() == "":
        dq_records.append([row["invoice_id"], "Missing description"])
    if pd.isna(row["po_id"]):
        dq_records.append([row["invoice_id"], "Missing po_id"])

data_quality_issues = pd.DataFrame(dq_records, columns=["invoice_id", "issue_reason"]).drop_duplicates()

print("Data quality checks completed.")
display(data_quality_issues.head(20))

Data quality checks completed.


,invoice_id,issue_reason
0,INV00004,Missing vendor_id
1,INV00008,Invalid amount
2,INV00011,Missing description
3,INV00026,Missing po_id
